# Aktivitas Naive Bayes — Apakah Cocok Bermain Tenis Hari Ini?

**Mata Kuliah:** DSB07 — Machine Learning  
**Topik:** Naive Bayes (Pertemuan 10)  
**Durasi:** 45 menit  

## Anggota Kelompok
_Isi nama & NIM di bawah ini sebelum mulai:_
1. NELSON CAHYADI - 32230122  
2. PASKALIS PERTOMO LIDUN NUNANG - 32230099  
3. NATHANAEL CHRISTIAN GAZALI - 32230090  
4. DAVID SANTOSO - 32230144  
5. DENNIS CHRISTAN - 32230123

## Petunjuk
Lengkapi setiap sel berlabel `# TODO`. Jangan ubah sel yang sudah berisi kode lengkap kecuali diminta.
Jalankan sel secara berurutan dari atas ke bawah.

## Setup

In [32]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option("display.precision", 4)
np.set_printoptions(precision=4, suppress=True)

## Tahap 1 — Memahami Data (5 menit)

Dataset `play_tennis` berisi 14 hari pengamatan apakah cocok untuk bermain tenis berdasarkan kondisi cuaca.

| Fitur | Nilai mungkin |
|-------|---------------|
| Outlook | Sunny, Overcast, Rain |
| Temperature | Hot, Mild, Cool |
| Humidity | High, Normal |
| Wind | Weak, Strong |
| **Play** (label) | **Yes, No** |

In [33]:
# Ganti USERNAME dengan akun GitHub dosen Anda setelah repo di-push.
url = "https://raw.githubusercontent.com/ubm-ml/naive-bayes/main/data/play_tennis.csv"
df = pd.read_csv(url)
df

,Outlook,Temperature,Humidity,Wind,Play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [34]:
# TODO 1: Tampilkan distribusi label 'Play' (berapa Yes, berapa No)
# TODO 1: Tampilkan distribusi label 'Play'
df["Play"].value_counts()


,count
Play,
Yes,9
No,5


## Tahap 2 — Hitung Manual (15 menit)

**Kasus uji:** Hari ini cuaca **Sunny**, suhu **Cool**, kelembapan **High**, angin **Strong**.  
**Pertanyaan:** Apakah cocok bermain tenis (`Play = Yes` atau `No`)?

Rumus (slide 2.2):
$$P(K \mid F_1, F_2, \dots, F_n) \propto P(K) \times \prod_{i=1}^{n} P(F_i \mid K)$$

Kita akan menghitung *tanpa* Laplace smoothing terlebih dahulu (sesuai rumus di slide).

In [35]:
test = {"Outlook": "Sunny", "Temperature": "Cool", "Humidity": "High", "Wind": "Strong"}
test

{'Outlook': 'Sunny',
 'Temperature': 'Cool',
 'Humidity': 'High',
 'Wind': 'Strong'}

### 2.1 Probabilitas Prior P(K)

In [36]:
# TODO 2: Hitung P(Yes) dan P(No)
n_total = len(df)
n_yes = (df["Play"] == "Yes").sum()
n_no = (df["Play"] == "No").sum()

p_yes = n_yes / n_total
p_no = n_no / n_total

print(f"P(Yes) = {p_yes:.4f}")
print(f"P(No) = {p_no:.4f}")


P(Yes) = 0.6429
P(No) = 0.3571


### 2.2 Probabilitas Kondisional P(F_i | K)

Untuk setiap fitur dalam kasus uji, hitung berapa kali nilainya muncul dalam tiap kelas, dibagi total baris kelas tersebut.

In [37]:
df_yes = df[df["Play"] == "Yes"]
df_no = df[df["Play"] == "No"]

# Untuk kelas Yes
p_sunny_yes  = (df_yes["Outlook"] == "Sunny").sum() / len(df_yes)
p_cool_yes   = (df_yes["Temperature"] == "Cool").sum() / len(df_yes)
p_high_yes   = (df_yes["Humidity"] == "High").sum() / len(df_yes)
p_strong_yes = (df_yes["Wind"] == "Strong").sum() / len(df_yes)

# Untuk kelas No
p_sunny_no  = (df_no["Outlook"] == "Sunny").sum() / len(df_no)
p_cool_no   = (df_no["Temperature"] == "Cool").sum() / len(df_no)
p_high_no   = (df_no["Humidity"] == "High").sum() / len(df_no)
p_strong_no = (df_no["Wind"] == "Strong").sum() / len(df_no)

print("Kelas Yes:", p_sunny_yes, p_cool_yes, p_high_yes, p_strong_yes)
print("Kelas No :", p_sunny_no, p_cool_no, p_high_no, p_strong_no)


Kelas Yes: 0.2222222222222222 0.3333333333333333 0.3333333333333333 0.3333333333333333
Kelas No : 0.6 0.2 0.8 0.6


### 2.3 Posterior & Keputusan

In [38]:
# Kasus uji
test = {"Outlook": "Sunny", "Temperature": "Cool", "Humidity": "High", "Wind": "Strong"}

# TODO 5: Hitung skor posterior (proporsional)
score_yes = p_yes * p_sunny_yes * p_cool_yes * p_high_yes * p_strong_yes
score_no  = p_no  * p_sunny_no  * p_cool_no  * p_high_no  * p_strong_no

# TODO 6: Normalisasi
total = score_yes + score_no
post_yes = score_yes / total
post_no  = score_no / total

prediksi_manual = "Yes" if post_yes > post_no else "No"

print(f"P(Yes | x) = {post_yes:.4f}")
print(f"P(No | x) = {post_no:.4f}")
print(f"Prediksi manual: {prediksi_manual}")


P(Yes | x) = 0.2046
P(No | x) = 0.7954
Prediksi manual: No


## Tahap 3 — Implementasi dengan scikit-learn (15 menit)

`CategoricalNB` di sklearn membutuhkan input numerik, jadi fitur kategorikal kita ubah dulu dengan `OrdinalEncoder`.

In [39]:
features = ["Outlook", "Temperature", "Humidity", "Wind"]

encoder = OrdinalEncoder()
X = encoder.fit_transform(df[features])
y = df["Play"].values

print("Pemetaan kategori -> angka:")
for col, cats in zip(features, encoder.categories_):
    print(f"  {col}: {dict(enumerate(cats))}")

Pemetaan kategori -> angka:
  Outlook: {0: 'Overcast', 1: 'Rain', 2: 'Sunny'}
  Temperature: {0: 'Cool', 1: 'Hot', 2: 'Mild'}
  Humidity: {0: 'High', 1: 'Normal'}
  Wind: {0: 'Strong', 1: 'Weak'}


In [40]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score
import pandas as pd

# Encode fitur kategorikal
features = ["Outlook", "Temperature", "Humidity", "Wind"]
encoder = OrdinalEncoder()
X = encoder.fit_transform(df[features])
y = df["Play"]

# Latih model Naive Bayes dengan Laplace smoothing (alpha=1)
model = CategoricalNB(alpha=1)
model.fit(X, y)

y_pred = model.predict(X)
akurasi = accuracy_score(y, y_pred)

print(f"Akurasi pada data latih: {akurasi:.4f}")


Akurasi pada data latih: 0.9286


In [41]:
# Kasus uji sama seperti sebelumnya
test = {"Outlook": "Sunny", "Temperature": "Cool", "Humidity": "High", "Wind": "Strong"}

# Ubah ke DataFrame dan transformasi dengan encoder
x_test_df = pd.DataFrame([test])
x_test = encoder.transform(x_test_df[features])

# Prediksi dan probabilitas
prediksi_sklearn = model.predict(x_test)
proba_sklearn = model.predict_proba(x_test)

print(f"Prediksi sklearn : {prediksi_sklearn[0]}")
print(f"Kelas-kelas      : {model.classes_}")
print(f"Probabilitas     : {proba_sklearn[0]}")


Prediksi sklearn : No
Kelas-kelas      : ['No' 'Yes']
Probabilitas     : [0.7201 0.2799]


## Tahap 4 — Diskusi Kelompok (7 menit)

_Tulis jawaban kelompok dalam sel di bawah._

1. Apakah **prediksi kelas** manual sama dengan sklearn? Apakah **nilai probabilitas**-nya juga sama? Jika berbeda, kira-kira kenapa?  
   _Hint: cari arti parameter `alpha` di `CategoricalNB` (Laplace / additive smoothing)._
2. Coba ubah `CategoricalNB(alpha=1e-10)` dan jalankan ulang Tahap 3. Apakah probabilitas sklearn jadi lebih dekat ke hasil manual?
3. Apa yang akan terjadi jika kasus uji mengandung kombinasi (kategori, kelas) yang **tidak pernah muncul** di data latih (misalnya `Outlook=Overcast` pada kelas `No`)? Bagaimana Laplace smoothing menyelamatkan situasi ini?  
   _Hint: lihat slide 1.3 poin #3 (Sensitif terhadap Atribut yang Hilang)._
4. Naive Bayes mengasumsikan **semua fitur saling independen**. Menurut kelompok Anda, apakah `Humidity` dan `Outlook` benar-benar independen di dunia nyata? Apa konsekuensinya untuk akurasi model?

**Jawaban Kelompok:**

1. - Prediksi kelas: Sama-sama menghasilkan “No”, jadi model dan perhitungan manual sejalan.  
   - Nilai probabilitas: sedikit berbeda, manual P(No|x) = 0.7954, sedangkan sklearn P(No|x) = 0.7201
   - Alasan perbedaan:  
   parameter alpha di CategoricalNB menerapkan Laplace smoothing, yaitu menambahkan nilai kecil (biasanya 1) ke setiap frekuensi kategori agar tidak ada probabilitas nol.
   Ini membuat distribusi lebih “halus” dan menghindari pembagian nol, tapi juga menggeser sedikit nilai probabilitas dibanding perhitungan manual tanpa smoothing.
2. Probabilitas sklearn akan lebih dekat ke hasil manual, karena model menghitung frekuensi murni tanpa penyesuaian tambahan.  

3. Jika ada kombinasi kategori dan kelas yang tidak pernah muncul di data latih, probabilitasnya jadi nol sehingga seluruh perhitungan posterior untuk kelas itu ikut nol. Laplace smoothing menambahkan nilai kecil ke setiap kemungkinan kategori sehingga tidak ada probabilitas nol, jadi model tetap bisa menghitung meskipun kombinasi baru muncul.

4. Naive Bayes menganggap semua fitur independen, padahal di dunia nyata seperti Humidity dan Outlook sering berkorelasi. Karena asumsi ini tidak sepenuhnya benar, model bisa menghasilkan probabilitas gabungan yang kurang akurat dan akurasi prediksi bisa menurun pada data nyata.

## Tahap 5 — Refleksi (3 menit)

Tulis 3 kalimat singkat:
- Hal terpenting yang saya pelajari hari ini adalah Naive Bayes bisa menghitung probabilitas kelas dengan sederhana, dan hasil manual bisa dibandingkan langsung dengan model sklearn.
- Bagian paling sulit adalah memahami efek Laplace smoothing (alpha) karena ia mengubah sedikit nilai probabilitas dari hasil manual.
- Naive Bayes cocok dipakai ketika dataset kecil, fitur kategorikal, dan asumsi independensi antar fitur cukup masuk akal sehingga model tetap efisien dan akurat.

## Submission
Simpan notebook ini ke Google Drive masing-masing, lalu kumpulkan link **share** (akses *Anyone with the link → Viewer*) ke kanal kelas yang ditentukan dosen.